In [1]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

True

In [3]:
class SubState(TypedDict):
    input_text:str
    translated_text:str

In [4]:
subgraph_model = ChatGroq(model="openai/gpt-oss-120b", api_key=os.getenv("GROQ_API_KEY"))

In [ ]:
def translate_subgraph(state:SubState) -> SubState:
    prompt = f"Translate the following text to Japanese: {state['input_text']}"
    translated_text = subgraph_model.invoke(prompt).content
    return { "translated_text": translated_text}

In [7]:
subgraph_builder = StateGraph(SubState)

subgraph_builder.add_node("translate_subgraph", translate_subgraph)

subgraph_builder.add_edge(START, "translate_subgraph")
subgraph_builder.add_edge("translate_subgraph", END)

subgraph = subgraph_builder.compile();


In [8]:
class ParentState(TypedDict):
    user_query:str
    text_english:str
    text_japanese:str

In [9]:
parent_model = ChatGroq(model="openai/gpt-oss-120b", api_key=os.getenv("GROQ_API_KEY"))

In [10]:
def generate_answer(state:ParentState) -> ParentState:
    prompt = f"You are a helpful assistant. Answer the user's query: {state['user_query']}."
    answer = parent_model.invoke(prompt).content
    return { "text_english": answer}



In [11]:
def translate_answer(state:ParentState) -> ParentState:
    subgraph_result = subgraph.invoke({"input_text": state["text_english"]})
    return {"text_japanese": subgraph_result["translated_text"]}

In [12]:
parent_graph_builder = StateGraph(ParentState)

parent_graph_builder.add_node("generate_answer", generate_answer)
parent_graph_builder.add_node("translate_answer", translate_answer)

parent_graph_builder.add_edge(START, "generate_answer")
parent_graph_builder.add_edge("generate_answer", "translate_answer")
parent_graph_builder.add_edge("translate_answer", END)

parent_graph = parent_graph_builder.compile()

In [13]:
parent_graph.invoke({"user_query": "Explain quantum computing in simple terms."})

{'user_query': 'Explain quantum computing in simple terms.',
 'text_english': '**Quantum Computing in Simple Terms**\n\n---\n\n### 1. The basic idea\n- **Classical computers** (the ones we use every day) store information as bits that are either **0** or **1**.\n- **Quantum computers** use **qubits** (quantum bits). A qubit can be **0, 1, or both at the same time** because of a quantum property called **superposition**.\n\n---\n\n### 2. Key quantum tricks\n\n| Quantum concept | What it means (in plain language) | Why it helps a computer |\n|-----------------|-----------------------------------|--------------------------|\n| **Superposition** | Imagine a spinning coin that is simultaneously “heads” *and* “tails” until you look at it. A qubit can hold many possibilities at once. | One qubit can represent many numbers at the same time, so a small quantum computer can explore many solutions in parallel. |\n| **Entanglement** | Two qubits become linked, like a pair of dice that always show 